# dart_korea_fs_loader_v7

**v6 → v7 변경 (2026-08)**

| 항목 | v6 | v7 |
|---|---|---|
| 테이블 | `korea_fs_data_from_DART_V2` | **`korea_fs_data_from_DART_V3`** (V2 보존) |
| PK | `(corp_code, bsns_year, reprt_code, quarter, account_id)` | **`(corp_code, bsns_year, reprt_code, sj_div, ord)`** |
| 추가 컬럼 | — | `ord`(보고서 내 행 순번), `thstrm_add_amount`(누적금액), `fs_div`(CFS/OFS), `loaded_at` |
| 정렬 | account_nm | sj_div, ord (원본 순서) |

**왜:** v6 PK 에 `sj_div`·`account_nm` 이 없어서 한 보고서 안의 `-표준계정코드 미사용-` 행들이 같은 키로 충돌 → `ON DUPLICATE KEY UPDATE` 가 앞 행을 덮어써 **첫 행만 남음**. 에이피알 2026Q1/H1 영업이익·당기순이익이 이렇게 사라졌고(id 8634211·8634226 소실), 모든 종목의 `미사용` 다중 보고서가 같은 피해를 입었음. `ord` 는 DART 가 부여하는 행 순번이라 계정 ID/명이 반복돼도 안전.

**재적재:** V3 는 빈 테이블이므로 Cell 6(전체 재수집, `START_YEAR=2015`) 실행. 일일 한도(status 020) 에 걸리면 다음날 Cell 3(증분) 으로 이어서. 재적재 후 v7 검증 셀의 `compare_v2_v3(db_info, "278470", 2026)` 로 복구 확인.

**하류:** 밸류에이션 노트북(`dart_fcff_valuation_v1`, `dart_rim_valuation_v1`) 은 Control Panel 의 `TABLE_DART_FS = "korea_fs_data_from_DART_V3"` 만 변경. 컬럼은 상위 호환(기존 컬럼 유지).


In [1]:
import os
import sys
import io
import time
import logging
import requests
import zipfile
import xml.etree.ElementTree as ET
from typing import Optional, Dict, List
from pathlib import Path
import pandas as pd
import datetime as dt
import pymysql

# ---------------------------------------------------------
# 기본 로깅 설정
# ---------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger(__name__)


# ---------------------------------------------------------
# 저장 대상 테이블 (V1 -> V2 전환용 단일 스위치)
# - "korea_fs_data_from_DART"    : 기존 V1 테이블
# - "korea_fs_data_from_DART_V2" : V2 테이블 (현재 기본값)
# ---------------------------------------------------------
FS_TABLE = "korea_fs_data_from_DART_V3"   # ★ v7: PK 재정의로 신규 테이블 (V2 는 보존)

# ★ v7: 저장 컬럼 (ord / thstrm_add_amount / fs_div 추가)
TARGET_COLS = [
    'corp_code', 'bsns_year', 'reprt_code', 'sj_div', 'sj_nm', 'ord',
    'account_id', 'account_nm', 'thstrm_nm', 'thstrm_amount', 'thstrm_add_amount',
    'fs_div', 'quarter', 'report_date',
]


# ---------------------------------------------------------
# 0) 프로젝트 루트 자동 탐색 (DATA 폴더 기준)
# ---------------------------------------------------------
def add_repo_path():
    """프로젝트 루트를 자동 탐색하여 sys.path에 추가"""
    if '__file__' in globals():
        current = Path(__file__).resolve().parent
    else:
        current = Path.cwd()

    for parent in [current] + list(current.parents):
        if (parent / "DATA").exists():
            if str(parent) not in sys.path:
                sys.path.insert(0, str(parent))
            logger.info(f"Project root added: {parent}")
            return str(parent)

    fallback = r"C:\Users\Hoyoung_Park\PyCharmMiscProject\stock_forecast"
    if os.path.isdir(fallback):
        if fallback not in sys.path:
            sys.path.insert(0, fallback)
        logger.warning(f"Using fallback path: {fallback}")
        return fallback

    raise FileNotFoundError("DATA 폴더를 찾을 수 없습니다.")


try:
    project_root = add_repo_path()
    from DATA.stock_invest_function import get_db_host
except ImportError:
    logger.warning("stock_invest_function import 실패 - DB 정보를 직접 설정해야 합니다")


# ---------------------------------------------------------
# 1) corp_code 목록 불러오기 (DART corpCode.xml)
# ---------------------------------------------------------
def load_corp_code(api_key: str) -> pd.DataFrame:
    """
    DART에서 corpCode.zip을 내려받아
    corp_code, corp_name, stock_code 정보를 DataFrame으로 반환.
    """
    url = "https://opendart.fss.or.kr/api/corpCode.xml"
    params = {"crtfc_key": api_key}

    # corpCode는 한 번만 부르므로 retry로 안정성 확보
    last_exc = None
    r = None
    for attempt in range(3):
        try:
            r = requests.get(url, params=params, timeout=60)
            r.raise_for_status()
            break
        except requests.exceptions.RequestException as e:
            last_exc = e
            if attempt < 2:
                wait = 2 ** attempt
                logger.warning(f"[RETRY corpCode {attempt+1}/3] ({type(e).__name__}) wait={wait}s")
                time.sleep(wait)
    if r is None:
        raise RuntimeError(f"corpCode.xml 호출 3회 실패: {last_exc}")

    content_type = (r.headers.get("Content-Type") or "").lower()
    head_bytes = r.content[:4]  # ZIP 여부 판별용 (b'PK\\x03\\x04')

    # 1) 에러(XML) 응답인지 먼저 체크
    if ("xml" in content_type or "text" in content_type) and not head_bytes.startswith(b"PK"):
        try:
            root = ET.fromstring(r.text)
            status = root.findtext("status")
            message = root.findtext("message")
            if status != "000":
                raise RuntimeError(
                    f"[DART corpCode 오류] status={status}, message={message}"
                )
        except ET.ParseError:
            raise RuntimeError(
                f"[DART corpCode 오류] XML 파싱 실패. "
                f"Content-Type={content_type}, text={r.text[:200]}"
            )

    # 2) 정상: ZIP 파일 처리
    with zipfile.ZipFile(io.BytesIO(r.content)) as z:
        xml_name = None
        for name in z.namelist():
            if name.lower().endswith(".xml"):
                xml_name = name
                break

        if xml_name is None:
            raise RuntimeError(
                f"[DART corpCode 오류] ZIP 안에 XML 파일이 없습니다. files={z.namelist()}"
            )

        with z.open(xml_name) as xml_file:
            tree = ET.parse(xml_file)
            root = tree.getroot()

    # 3) XML → DataFrame 변환
    rows = []
    for child in root.findall("list"):
        corp_code = child.findtext("corp_code")
        corp_name = child.findtext("corp_name")
        stock_code = child.findtext("stock_code")
        rows.append(
            {
                "corp_code": corp_code,
                "corp_name": corp_name,
                "stock_code": stock_code,
            }
        )

    df = pd.DataFrame(rows)
    df = df[df["stock_code"].notnull() & (df["stock_code"] != "")]
    df.reset_index(drop=True, inplace=True)
    return df


# ---------------------------------------------------------
# 2) FinanceDataReader 종목 코드로 corp_code 찾기
# ---------------------------------------------------------
def get_corp_info(corp_df: pd.DataFrame, stock_code: str) -> Optional[Dict]:
    """
    FinanceDataReader 형식의 종목코드(예: '005930')로
    corp_df에서 해당 기업의 corp_code, corp_name, stock_code 를 찾아 dict로 반환.
    """
    row = corp_df.loc[corp_df["stock_code"] == stock_code]
    if row.empty:
        return None

    row = row.iloc[0]
    return {
        "corp_code": row["corp_code"],
        "corp_name": row["corp_name"],
        "stock_code": row["stock_code"],
    }

def test_db_connection(db_info: dict) -> bool:
    """
    MariaDB 연결 테스트 함수.
    연결 성공하면 True, 실패하면 False 반환.
    """
    try:
        conn = pymysql.connect(
            host=db_info["host"],
            port=db_info["port"],
            user=db_info["user"],
            password=db_info["password"],
            database=db_info["database"],
            charset="utf8mb4",
            connect_timeout=5
        )
        conn.close()
        logger.info("DB 연결 성공")
        return True
    except Exception as e:
        logger.error(f"DB 연결 실패: {e}")
        return False



# ---------------------------------------------------------
# 3) 분기별 재무제표 수신 (fnlttSinglAcntAll)
#    - 먼저 CFS 시도, 없으면 OFS로 fallback
# ---------------------------------------------------------
def get_dart_fs_quarterly(api_key: str,
                          corp_code: str,
                          start_year: int,
                          end_year: int) -> pd.DataFrame:
    """
    DART 'fnlttSinglAcntAll' API를 사용하여 분기별 재무제표 수집.
    먼저 CFS(연결) 시도 → 자료 없으면 OFS(개별)로 자동 fallback.
    """

    def fetch_one_year(api_key, corp_code, year, fs_div):
        """특정 연도·fs_div로 조회하는 내부 함수 (throttling + retry 포함)"""
        url = "https://opendart.fss.or.kr/api/fnlttSinglAcntAll.json"
        reprt_map = {
            "11013": ("Q1", "-03-31"),
            "11012": ("H1", "-06-30"),
            "11014": ("Q3", "-09-30"),
            "11011": ("FY", "-12-31"),
        }

        rows: List[Dict] = []

        for reprt_code, (quarter_label, date_suffix) in reprt_map.items():
            params = {
                "crtfc_key": api_key,
                "corp_code": corp_code,
                "bsns_year": str(year),
                "reprt_code": reprt_code,
                "fs_div": fs_div,
            }

            # 분당 1,000회 제한 대비 throttling (호출당 0.08초 = 분당 최대 ~750회)
            time.sleep(0.08)

            # Transient 에러 대응: 최대 3회 재시도 (exponential backoff)
            data = None
            last_exc = None
            for attempt in range(3):
                try:
                    r = requests.get(url, params=params, timeout=30)
                    r.raise_for_status()
                    data = r.json()
                    break
                except (requests.exceptions.RequestException, ValueError) as e:
                    last_exc = e
                    if attempt < 2:
                        wait = 2 ** attempt  # 1s, 2s
                        logger.warning(
                            f"[RETRY {attempt+1}/3] corp={corp_code} year={year} "
                            f"reprt={reprt_code} ({type(e).__name__}) wait={wait}s"
                        )
                        time.sleep(wait)

            if data is None:
                # 3회 모두 실패 → 이 보고서만 skip (전체 중단하지 않음)
                logger.error(
                    f"[SKIP] corp={corp_code} year={year} reprt={reprt_code}: "
                    f"{type(last_exc).__name__}: {last_exc}"
                )
                continue

            status = data.get("status")
            if status != "000":
                # "013" = 자료 없음, "020" = 사용자 호출 초과 등
                if status == "020":
                    # 일일 한도(20,000건) 초과는 대기로 해결되지 않음 → 즉시 중단
                    raise RuntimeError(
                        f"[RATE LIMIT] DART 일일 호출 한도 초과 (status=020). "
                        f"corp={corp_code} year={year} reprt={reprt_code}. "
                        f"내일 다시 실행하거나 다른 API 키를 사용하세요."
                    )
                continue   # status 013 등 자료 없음 → 다음 보고서

            for item in data.get("list", []):
                row = {
                    "corp_code": item.get("corp_code"),
                    "bsns_year": int(item.get("bsns_year")),
                    "reprt_code": item.get("reprt_code"),
                    "sj_div": item.get("sj_div"),
                    "sj_nm": item.get("sj_nm"),
                    "account_id": item.get("account_id"),
                    "account_nm": item.get("account_nm"),
                    "thstrm_nm": item.get("thstrm_nm"),
                    "thstrm_amount": item.get("thstrm_amount"),
                    "thstrm_add_amount": item.get("thstrm_add_amount"),   # ★ v7 누적금액
                    "ord": item.get("ord"),                               # ★ v7 보고서 내 행 순번 (PK)
                    "fs_div": fs_div,                                     # ★ v7 CFS/OFS
                    "quarter": quarter_label,
                }
                # 날짜
                try:
                    row["report_date"] = dt.datetime.strptime(
                        f"{year}{date_suffix}", "%Y-%m-%d"
                    ).date()
                except Exception:
                    row["report_date"] = None

                rows.append(row)

        return rows

    # 1) CFS 먼저
    all_rows: List[Dict] = []
    for year in range(start_year, end_year + 1):
        rows = fetch_one_year(api_key, corp_code, year, fs_div="CFS")
        if rows:
            all_rows.extend(rows)

    # 2) CFS 없으면 OFS로 재시도
    if len(all_rows) == 0:
        print("[INFO] CFS(연결) 자료 없음 → OFS(개별)로 재시도합니다.")
        for year in range(start_year, end_year + 1):
            rows = fetch_one_year(api_key, corp_code, year, fs_div="OFS")
            if rows:
                all_rows.extend(rows)

    if not all_rows:
        print("[WARN] CFS/OFS 모두 자료 없음")
        return pd.DataFrame()

    df = pd.DataFrame(all_rows)

    # 금액 숫자 변환
    df["thstrm_amount"] = pd.to_numeric(df["thstrm_amount"], errors="coerce")
    df["thstrm_add_amount"] = pd.to_numeric(df["thstrm_add_amount"], errors="coerce")
    df["ord"] = pd.to_numeric(df["ord"], errors="coerce")

    df = df.sort_values(["bsns_year", "reprt_code", "sj_div", "ord"]).reset_index(drop=True)
    return df

def run_dart_fs_for_top_n(api_key: str,
                          db_info: dict,
                          start_year: int = 2015,
                          end_year: int = 2025,
                          top_n: int = 50,
                          batch_size: int = 10,
                          table_name: str = FS_TABLE):
    """
    DART 상장사 목록에서 stock_code 오름차순으로 앞쪽 top_n개 기업의
    분기 재무 데이터를 수집한다.

    주의:
        시가총액 기반 정렬은 더 이상 수행하지 않는다.
        시총 기반 필터링이 필요하면 호출부에서 ticker 리스트를 준비한 뒤
        run_dart_fs_for_stock_list()를 직접 호출할 것.

    반환:
        error_list: [(stock_code, corp_name, 에러메시지), ...]
    """

    if not test_db_connection(db_info):
        logger.error("DB 연결 실패로 작업을 중단합니다")
        return []
    logger.info("DB 연결 테스트 완료")

    print("=" * 70)
    logger.info("[STEP 1] DART 기업 목록 로드 중...")
    corp_df = load_corp_code(api_key)

    # 상장사만 필터링 + stock_code 정규화
    corp_df = corp_df[
        corp_df["stock_code"].notna() &
        (corp_df["stock_code"] != "") &
        (corp_df["stock_code"].str.strip() != "")
    ].copy()
    corp_df["stock_code"] = corp_df["stock_code"].astype(str).str.zfill(6)
    logger.info(f"DART 상장사 필터링 완료: {len(corp_df)}개")

    # stock_code 순 정렬 후 앞쪽 top_n개 선택
    corp_df = corp_df.sort_values("stock_code").reset_index(drop=True)
    before = len(corp_df)
    corp_df = corp_df.head(top_n).copy()
    logger.info(f"stock_code 순 상위 {top_n}개 선택: {before}개 -> {len(corp_df)}개")

    total_companies = len(corp_df)
    logger.info(f"최종 대상 기업(루프 대상): {total_companies}개")

    print("\n[상장사 샘플]")
    print(corp_df[["corp_name", "stock_code"]].head(10))
    print()

    # -------------------------------------------------
    # 3) 메인 루프: 회사별로 DART 재무제표 수집 + 배치 저장
    # -------------------------------------------------
    error_list = []
    batch_list: List[pd.DataFrame] = []
    batch_codes: List[str] = []

    target_cols = TARGET_COLS   # ★ v7

    processed_count = 0

    for idx, row in corp_df.iterrows():
        stock_code = row["stock_code"]
        corp_code = row["corp_code"]
        corp_name = row["corp_name"]

        logger.info(f"[{processed_count + 1}/{total_companies}] {corp_name}({stock_code}) 처리 중...")

        try:
            fs_df = get_dart_fs_quarterly(
                api_key=api_key,
                corp_code=corp_code,
                start_year=start_year,
                end_year=end_year,
            )

            if fs_df.empty:
                logger.warning(f"{corp_name}({stock_code}) : 재무데이터 없음 (fs_df empty)")
                processed_count += 1
                continue

            # 필요한 컬럼만 추출
            missing_cols = [c for c in target_cols if c not in fs_df.columns]
            if missing_cols:
                logger.warning(f"{corp_name}({stock_code}) : 필요한 컬럼 누락 - {missing_cols}")
                processed_count += 1
                continue

            fs_df_refined = fs_df[target_cols].copy()
            fs_df_refined["ticker"] = stock_code  # 이 batch 함수에서는 ticker를 미리 넣어둡니다.

            batch_list.append(fs_df_refined)
            batch_codes.append(stock_code)
            processed_count += 1

            # 배치 크기에 도달하면 DB에 저장
            if len(batch_list) >= batch_size:
                logger.info(f"[BATCH SAVE] 회사 {len(batch_list)}개 묶어서 DB 저장 시도...")
                try:
                    save_fs_batch_to_db(batch_list, db_info=db_info, table_name=table_name)
                    logger.info(f"[BATCH SAVE] 저장 완료 (회사 {len(batch_list)}개)")
                except Exception as be:
                    logger.error(f"[BATCH SAVE ERROR] 배치 저장 중 오류 발생: {be}")
                    # 배치에 포함된 종목 모두를 에러 리스트에 추가
                    for sc in batch_codes:
                        error_list.append((sc, "BATCH_ERROR", str(be)))
                finally:
                    # 배치 초기화
                    batch_list = []
                    batch_codes = []

        except Exception as e:
            logger.error(f"{corp_name}({stock_code}) 처리 중 오류 발생: {e}")
            error_list.append((stock_code, corp_name, str(e)))
            processed_count += 1
            continue

    # 마지막으로 남은 배치 처리
    if batch_list:
        logger.info(f"[FINAL BATCH SAVE] 남은 회사 {len(batch_list)}개 DB 저장 시도...")
        try:
            save_fs_batch_to_db(batch_list, db_info=db_info, table_name=table_name)
            logger.info(f"[FINAL BATCH SAVE] 저장 완료 (회사 {len(batch_list)}개)")
        except Exception as be:
            logger.error(f"[FINAL BATCH SAVE ERROR] 배치 저장 중 오류 발생: {be}")
            for sc in batch_codes:
                error_list.append((sc, "BATCH_ERROR", str(be)))

    logger.info(f"작업 완료. 총 기업 수: {total_companies}, 에러 기업 수: {len(error_list)}")

    if error_list:
        print("\n[에러 발생 종목 목록]")
        for sc, name, msg in error_list:
            print(f" - {sc} / {name} / {msg[:100]}")

    return error_list

def run_dart_fs_for_top_range(api_key: str,
                              db_info: dict,
                              start_year: int,
                              end_year: int,
                              top_start: int,
                              top_end: int,
                              batch_size: int = 10,
                              table_name: str = FS_TABLE):
    """
    DART 상장사 목록에서 stock_code 오름차순 [top_start, top_end) 구간의
    기업들에 대해 분기 재무 데이터를 수집한다.

    슬라이싱: Python 표준 half-open 구간
        top_start=0,    top_end=100   → 첫 100개
        top_start=100,  top_end=200   → 101~200번째
        top_start=0,    top_end=99999 → 전체 (안전하게 큰 값)

    반환:
        error_list: [(stock_code, corp_name, 에러메시지), ...]
    """
    if not test_db_connection(db_info):
        logger.error("DB 연결 실패로 작업 중단")
        return []

    corp_df = load_corp_code(api_key)
    corp_df = corp_df[corp_df["stock_code"].notnull()].copy()
    corp_df["stock_code"] = corp_df["stock_code"].astype(str).str.zfill(6)

    # stock_code 순 정렬 후 지정 범위 슬라이싱
    corp_df = corp_df.sort_values("stock_code").reset_index(drop=True)
    total_before = len(corp_df)
    corp_df = corp_df.iloc[top_start:top_end].copy()

    logger.info(
        f"DART corp 리스트 [{top_start}:{top_end}) 구간 선택: "
        f"전체 {total_before}개 -> 대상 {len(corp_df)}개"
    )
    print(f"[INFO] stock_code 정렬 기준 {top_start}~{top_end-1}번째 기업 수: {len(corp_df)}")

    error_list = run_dart_fs_for_stock_list(
        api_key=api_key,
        db_info=db_info,
        stock_code_list=list(corp_df["stock_code"]),
        start_year=start_year,
        end_year=end_year,
        batch_size=batch_size,
        table_name=table_name,
    )
    return error_list

# ---------------------------------------------------------
# 4) DB 저장 함수
# ---------------------------------------------------------
def save_fs_batch_to_db(batch_list: List[pd.DataFrame],
                        db_info: dict,
                        table_name: str = FS_TABLE):
    """
    ★ v7: 배치 DataFrame 을 DB 에 저장.
      - PK = (corp_code, bsns_year, reprt_code, sj_div, ord)
        · v6 PK (…, quarter, account_id) 는 sj_div·account_nm 이 없어서 한 보고서 안의
          '-표준계정코드 미사용-' 행들이 서로 덮어쓰며 유실됐음 (예: 에이피알 2026 영업이익·당기순이익)
        · ord 는 DART 가 부여한 보고서 내 행 순번 → 같은 account_id/account_nm 이 반복돼도 안전
      - thstrm_add_amount(누적금액), fs_div(CFS/OFS) 추가 저장
    """
    if not batch_list:
        return

    df = pd.concat(batch_list, ignore_index=True)

    # 타입 정리
    df["bsns_year"] = pd.to_numeric(df["bsns_year"], errors="coerce").astype("Int64")
    df["quarter"] = df["quarter"].astype(str)
    df["reprt_code"] = df["reprt_code"].astype(str)
    df["thstrm_amount"] = pd.to_numeric(df["thstrm_amount"], errors="coerce")
    df["thstrm_add_amount"] = pd.to_numeric(df.get("thstrm_add_amount"), errors="coerce")
    df["ord"] = pd.to_numeric(df.get("ord"), errors="coerce")
    df["report_date"] = pd.to_datetime(df["report_date"], errors="coerce").dt.date
    df["account_id"] = df["account_id"].fillna("").astype(str)
    df["account_nm"] = df["account_nm"].fillna("").astype(str)
    df["sj_div"] = df["sj_div"].fillna("").astype(str)
    if "fs_div" not in df.columns:
        df["fs_div"] = None

    # ord 결측 시 보고서·재무제표 내 순번으로 대체 (PK 보호)
    miss = df["ord"].isna()
    if miss.any():
        logger.warning(f"[BATCH] ord 결측 {int(miss.sum())}행 → 순번으로 대체")
        df.loc[miss, "ord"] = (
            df[miss].groupby(["corp_code", "bsns_year", "reprt_code", "sj_div"]).cumcount() + 100000
        )
    df["ord"] = df["ord"].astype(int)

    # 동일 PK 가 배치 안에 중복되면 마지막 행 유지 (CFS/OFS 재시도 등)
    before = len(df)
    df = df.drop_duplicates(["corp_code", "bsns_year", "reprt_code", "sj_div", "ord"], keep="last")
    if before != len(df):
        logger.warning(f"[BATCH] 배치 내 PK 중복 제거: {before - len(df)} rows")

    # NaN/NaT → None
    df = df.astype(object).where(df.notnull(), None)

    conn = pymysql.connect(
        host=db_info["host"],
        port=db_info["port"],
        user=db_info["user"],
        password=db_info["password"],
        database=db_info["database"],
        charset="utf8mb4",
    )
    try:
        with conn.cursor() as cur:
            create_sql = f"""
            CREATE TABLE IF NOT EXISTS {table_name} (
                id                 BIGINT AUTO_INCREMENT,
                corp_code          VARCHAR(20)   NOT NULL,
                bsns_year          INT           NOT NULL,
                reprt_code         VARCHAR(10)   NOT NULL,
                sj_div             VARCHAR(10)   NOT NULL,
                ord                INT           NOT NULL,

                quarter            VARCHAR(10)   NOT NULL,
                sj_nm              VARCHAR(100),
                account_id         VARCHAR(150),
                account_nm         VARCHAR(255),
                thstrm_nm          VARCHAR(50),
                thstrm_amount      DOUBLE,
                thstrm_add_amount  DOUBLE,
                fs_div             VARCHAR(5),
                report_date        DATE,
                ticker             VARCHAR(20)   NOT NULL,
                loaded_at          TIMESTAMP DEFAULT CURRENT_TIMESTAMP ON UPDATE CURRENT_TIMESTAMP,

                PRIMARY KEY (corp_code, bsns_year, reprt_code, sj_div, ord),
                UNIQUE KEY uq_id (id),
                INDEX idx_ticker_year (ticker, bsns_year),
                INDEX idx_account_id (account_id),
                INDEX idx_ticker_sj (ticker, sj_div)
            ) CHARACTER SET utf8mb4;
            """
            cur.execute(create_sql)

            insert_sql = f"""
            INSERT INTO {table_name} (
                corp_code, bsns_year, reprt_code, sj_div, ord, quarter, sj_nm,
                account_id, account_nm, thstrm_nm, thstrm_amount, thstrm_add_amount,
                fs_div, report_date, ticker
            ) VALUES (
                %(corp_code)s, %(bsns_year)s, %(reprt_code)s, %(sj_div)s, %(ord)s, %(quarter)s, %(sj_nm)s,
                %(account_id)s, %(account_nm)s, %(thstrm_nm)s, %(thstrm_amount)s, %(thstrm_add_amount)s,
                %(fs_div)s, %(report_date)s, %(ticker)s
            )
            ON DUPLICATE KEY UPDATE
                quarter           = VALUES(quarter),
                sj_nm             = VALUES(sj_nm),
                account_id        = VALUES(account_id),
                account_nm        = VALUES(account_nm),
                thstrm_nm         = VALUES(thstrm_nm),
                thstrm_amount     = VALUES(thstrm_amount),
                thstrm_add_amount = VALUES(thstrm_add_amount),
                fs_div            = VALUES(fs_div),
                report_date       = VALUES(report_date),
                ticker            = VALUES(ticker);
            """

            records = df.to_dict(orient="records")
            cur.executemany(insert_sql, records)

        conn.commit()
        logger.info(f"[BATCH] {len(df)} rows saved into {table_name}")
    except Exception as e:
        conn.rollback()
        logger.error(f"[BATCH ERROR] {e}")
        raise
    finally:
        conn.close()


def run_dart_fs_for_stock_list(api_key: str,
                               db_info: dict,
                               stock_code_list: list,
                               start_year: int = 2015,
                               end_year: int = 2025,
                               batch_size: int = 10,
                               table_name: str = FS_TABLE):
    """
    지정한 stock_code 리스트(예: ['005930','000660', ...])에 대해서만
    DART 분기 재무제표를 수집하고, batch_size개 회사 단위로 DB에 저장.

    - api_key: DART API 키
    - db_info: MariaDB 접속 정보 딕셔너리
    - stock_code_list: 종목코드 리스트 (길이 N)
    - start_year, end_year: 재무제표 수집 연도 범위
    - batch_size: 몇 개 회사 단위로 DB에 저장할지 (기본 10)
    - table_name: 저장할 테이블 이름

    반환:
        error_list: [(stock_code, corp_name_or_reason, error_message), ...]
    """

    # 0) DB 연결 테스트
    if not test_db_connection(db_info):
        logger.error("DB 연결 실패로 작업을 중단합니다")
        return []

    logger.info("DB 연결 테스트 완료")

    # 1) DART 기업 목록 로드
    logger.info("[STEP 1] DART 기업 목록 로드 중...")
    corp_df = load_corp_code(api_key)

    # 상장사만 필터링 + stock_code 6자리 정규화
    corp_df = corp_df[
        corp_df["stock_code"].notna() &
        (corp_df["stock_code"] != "") &
        (corp_df["stock_code"].str.strip() != "")
    ].copy()
    corp_df["stock_code"] = corp_df["stock_code"].astype(str).str.zfill(6)

    logger.info(f"DART 상장사 필터링 완료: {len(corp_df)}개")

    # 2) 입력받은 stock_code 리스트 정규화 (중복 제거 + 6자리 패딩)
    normalized_codes = sorted(set(str(code).zfill(6) for code in stock_code_list))
    logger.info(f"사용자 지정 종목 수: {len(stock_code_list)}개 -> 정규화 후 {len(normalized_codes)}개")

    # corp_df에서 빠른 lookup을 위해 dict 생성 (stock_code -> (corp_code, corp_name))
    corp_map = {}
    for _, r in corp_df[["corp_code", "corp_name", "stock_code"]].iterrows():
        corp_map[r["stock_code"]] = (r["corp_code"], r["corp_name"])

    # 3) 메인 루프: 회사별 재무제표 수집 + 배치 저장
    error_list = []
    batch_list: List[pd.DataFrame] = []
    batch_codes: List[str] = []

    target_cols = TARGET_COLS   # ★ v7

    total = len(normalized_codes)
    processed = 0

    for stock_code in normalized_codes:
        processed += 1

        if stock_code not in corp_map:
            msg = "DART corp_code를 찾을 수 없음"
            logger.warning(f"[{processed}/{total}] {stock_code}: {msg}")
            error_list.append((stock_code, "NOT_FOUND_IN_DART", msg))
            continue

        corp_code, corp_name = corp_map[stock_code]
        logger.info(f"[{processed}/{total}] {corp_name}({stock_code}) 처리 중...")

        try:
            # 3-1) 재무데이터 수신
            fs_df = get_dart_fs_quarterly(
                api_key=api_key,
                corp_code=corp_code,
                start_year=start_year,
                end_year=end_year,
            )

            if fs_df.empty:
                msg = "재무데이터 없음 (fs_df empty)"
                logger.warning(f"{corp_name}({stock_code}) : {msg}")
                error_list.append((stock_code, corp_name, msg))
                continue

            # 3-2) 필요한 컬럼 체크
            missing_cols = [c for c in target_cols if c not in fs_df.columns]
            if missing_cols:
                msg = f"필요한 컬럼 누락: {missing_cols}"
                logger.warning(f"{corp_name}({stock_code}) : {msg}")
                error_list.append((stock_code, corp_name, msg))
                continue

            # 3-3) 정제 후 배치 리스트에 추가
            fs_df_refined = fs_df[target_cols].copy()
            fs_df_refined["ticker"] = stock_code  # 여기서 ticker 추가

            batch_list.append(fs_df_refined)
            batch_codes.append(stock_code)

            # 3-4) 배치 크기에 도달하면 DB에 저장
            if len(batch_list) >= batch_size:
                logger.info(f"[BATCH SAVE] 회사 {len(batch_list)}개 묶어서 DB 저장 시도...")
                try:
                    save_fs_batch_to_db(batch_list, db_info=db_info, table_name=table_name)
                    logger.info(f"[BATCH SAVE] 저장 완료 (회사 {len(batch_list)}개)")
                except Exception as be:
                    logger.error(f"[BATCH SAVE ERROR] 배치 저장 중 오류 발생: {be}")
                    for sc in batch_codes:
                        error_list.append((sc, "BATCH_ERROR", str(be)))
                finally:
                    batch_list = []
                    batch_codes = []

        except Exception as e:
            logger.error(f"{corp_name}({stock_code}) 처리 중 오류 발생: {e}")
            error_list.append((stock_code, corp_name, str(e)))
            continue

    # 4) 마지막으로 남은 배치 처리
    if batch_list:
        logger.info(f"[FINAL BATCH SAVE] 남은 회사 {len(batch_list)}개 DB 저장 시도...")
        try:
            save_fs_batch_to_db(batch_list, db_info=db_info, table_name=table_name)
            logger.info(f"[FINAL BATCH SAVE] 저장 완료 (회사 {len(batch_list)}개)")
        except Exception as be:
            logger.error(f"[FINAL BATCH SAVE ERROR] 배치 저장 중 오류 발생: {be}")
            for sc in batch_codes:
                error_list.append((sc, "BATCH_ERROR", str(be)))

    logger.info(f"작업 완료. 지정 종목 수: {total}, 에러 종목 수: {len(error_list)}")

    if error_list:
        print("\n[에러 발생 종목 목록]")
        for sc, name, msg in error_list:
            print(f" - {sc} / {name} / {msg[:100]}")

    return error_list

# =============================================================================
# DART 재무데이터 수집 - Ticker 리스트 입력 실행 스크립트 (수정버전)
# =============================================================================

import os
from typing import List



# DB 연결 함수 import (실패해도 셀 전체가 죽지 않도록 보호)
try:
    from DATA.stock_invest_function import get_db_host
except ImportError:
    logger.warning("stock_invest_function import 실패 - get_db_host 대신 기본 호스트 사용")
    def get_db_host():
        return "192.168.0.230"

def collect_dart_fs_by_tickers(
    ticker_list: List[str],
    api_key: str,
    start_year: int = 2025,
    end_year: int = 2025,
    batch_size: int = 10,
    table_name: str = FS_TABLE
):
    """
    ticker 리스트를 입력받아 DART 재무데이터를 수집하고 DB에 저장

    Parameters:
    -----------
    ticker_list : List[str]
        종목코드 리스트 (예: ['005930', '000660', '035420'])
    api_key : str
        DART API 키
    start_year : int
        수집 시작 연도 (기본: 2015)
    end_year : int
        수집 종료 연도 (기본: 2025)
    batch_size : int
        한 번에 저장할 회사 수 (기본: 10)
    table_name : str
        저장할 테이블명 (기본: korea_fs_data_from_DART_V2)

    Returns:
    --------
    error_list : list
        에러 발생 종목 리스트 [(ticker, corp_name, error_msg), ...]
    """

    # 1) DB 연결 정보 설정 - get_db_host() 함수 사용
    db_info = {
        'host': get_db_host(),
        'port': 3307,  # 포트 3307로 수정
        'user': 'stox7412',
        'password': 'Apt106503!~',
        'database': 'investar'
    }

    logger.info(f"DB 연결 정보: host={db_info['host']}, port={db_info['port']}, database={db_info['database']}")

    # 2) 입력 확인
    logger.info("=" * 70)
    logger.info("[재무데이터 수집 시작]")
    logger.info(f"대상 종목 수: {len(ticker_list)}개")
    logger.info(f"수집 기간: {start_year}년 ~ {end_year}년")
    logger.info(f"배치 크기: {batch_size}개")
    logger.info(f"저장 테이블: {table_name}")
    logger.info("=" * 70)

    # 샘플 출력
    if len(ticker_list) <= 10:
        logger.info(f"대상 종목: {ticker_list}")
    else:
        logger.info(f"대상 종목 샘플 (처음 10개): {ticker_list[:10]}")

    # 3) 재무데이터 수집 및 저장
    error_list = run_dart_fs_for_stock_list(
        api_key=api_key,
        db_info=db_info,
        stock_code_list=ticker_list,
        start_year=start_year,
        end_year=end_year,
        batch_size=batch_size,
        table_name=table_name
    )

    # 4) 결과 요약
    logger.info("=" * 70)
    logger.info("[작업 완료]")
    logger.info(f"총 대상 종목: {len(ticker_list)}개")
    logger.info(f"성공: {len(ticker_list) - len(error_list)}개")
    logger.info(f"에러 발생: {len(error_list)}개")

    if error_list:
        logger.warning("\n[에러 발생 종목 상세]")
        for ticker, name, msg in error_list:
            logger.warning(f"  - {ticker} ({name}): {msg[:100]}")
    else:
        logger.info("모든 종목 처리 완료!")

    logger.info("=" * 70)

    return error_list

2026-08-25 18:06:33 [INFO] Project root added: C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy


In [2]:
# ==========================================================
# 설정: API 키와 DB 정보
# ==========================================================
# ⚠️ 보안: API_KEY와 password는 git에 커밋되지 않도록 주의하세요.
#    가능하면 .env 파일이나 환경변수로 분리하는 것이 좋습니다.

API_KEY = "83658f1f91801354f1f3b29bc6eb4e3c53b41ae8"   # DART 오픈API 키

try:
    from DATA.stock_invest_function import get_db_host
    _db_host = get_db_host()
except ImportError:
    print("[WARN] stock_invest_function import 실패 → 기본 호스트(192.168.0.230) 사용")
    _db_host = "192.168.0.230"

db_info = {
    'host':     _db_host,
    'port':     3307,
    'user':     'stox7412',
    'password': 'Apt106503!~',
    'database': 'investar',
}

TABLE_NAME = "korea_fs_data_from_DART_V3"   # ★ v7 신규 테이블
TABLE_NAME_V2 = "korea_fs_data_from_DART_V2"   # 비교·검증용 (구 테이블)

print(f"API_KEY: {API_KEY[:8]}...{API_KEY[-4:]}")
print(f"DB host: {db_info['host']}")
print(f"저장 테이블: {TABLE_NAME}")


API_KEY: 83658f1f...1ae8
DB host: 192.168.0.230
저장 테이블: korea_fs_data_from_DART_V3


In [3]:
# ==========================================================
# ★ v7 검증: PK 충돌 유실 점검 + V2 대비 복구 행 확인
#  - 실행 전제: Cell 0, Cell 1
# ==========================================================
import pymysql
import pandas as pd


def _q(db_info, sql, params=None):
    conn = pymysql.connect(host=db_info["host"], port=db_info["port"], user=db_info["user"],
                           password=db_info["password"], database=db_info["database"], charset="utf8mb4")
    try:
        return pd.read_sql(sql, conn, params=params)
    finally:
        conn.close()


def verify_v7_vs_api(api_key: str, db_info: dict, stock_code: str, year: int,
                     table_name: str = TABLE_NAME) -> pd.DataFrame:
    """API 응답 행수 vs DB 행수 (보고서·재무제표별). 차이가 0 이어야 정상 (v6 에서는 '미사용' 충돌만큼 작았음)."""
    corp_df = load_corp_code(api_key)
    info = get_corp_info(corp_df, stock_code)
    if info is None:
        print(f"{stock_code}: corp_code 없음"); return pd.DataFrame()
    api_df = get_dart_fs_quarterly(api_key, info["corp_code"], year, year)
    if api_df.empty:
        print("API 자료 없음"); return pd.DataFrame()
    api_cnt = api_df.groupby(["reprt_code", "sj_div"]).size().rename("api_rows")
    db = _q(db_info, f"SELECT reprt_code, sj_div, COUNT(*) AS db_rows FROM {table_name} "
                     f"WHERE ticker=%s AND bsns_year=%s GROUP BY reprt_code, sj_div", (stock_code, year))
    out = api_cnt.reset_index().merge(db, on=["reprt_code", "sj_div"], how="left").fillna({"db_rows": 0})
    out["diff"] = out["api_rows"] - out["db_rows"]
    print(out.to_string(index=False))
    print("→ diff 합계:", int(out["diff"].sum()), "(0 이면 유실 없음)")
    return out


def compare_v2_v3(db_info: dict, stock_code: str, year: int,
                  v2: str = TABLE_NAME_V2, v3: str = TABLE_NAME) -> pd.DataFrame:
    """V3 에는 있고 V2 에는 없는 (sj_div, account_id, account_nm) — v6 에서 유실됐던 행."""
    a = _q(db_info, f"SELECT quarter, sj_div, account_id, account_nm, thstrm_amount FROM {v3} "
                    f"WHERE ticker=%s AND bsns_year=%s", (stock_code, year))
    b = _q(db_info, f"SELECT quarter, sj_div, account_id, account_nm FROM {v2} "
                    f"WHERE ticker=%s AND bsns_year=%s", (stock_code, year))
    key = ["quarter", "sj_div", "account_id", "account_nm"]
    m = a.merge(b.assign(_in_v2=1), on=key, how="left")
    lost = m[m["_in_v2"].isna()].drop(columns="_in_v2")
    print(f"[{stock_code} {year}] V3 {len(a)}행 / V2 {len(b)}행 / V2 에 없던 행 {len(lost)}개")
    if not lost.empty:
        print(lost.to_string(index=False))
    return lost


def check_unused_collisions(db_info: dict, table_name: str = TABLE_NAME, top: int = 20) -> pd.DataFrame:
    """보고서 안에 '-표준계정코드 미사용-' 행이 2개 이상인 케이스 — v6 라면 전부 유실 대상이었음."""
    df = _q(db_info, f"""
        SELECT ticker, bsns_year, quarter, COUNT(*) AS n_unused
        FROM {table_name}
        WHERE account_id LIKE '%%미사용%%'
        GROUP BY ticker, bsns_year, quarter
        HAVING n_unused >= 2
        ORDER BY n_unused DESC
    """)
    print(f"'미사용' 2개 이상 보고서: {len(df)}건 (상위 {top})")
    print(df.head(top).to_string(index=False))
    return df


# ---- 사용 예 (재적재 후) ----
# verify_v7_vs_api(API_KEY, db_info, "278470", 2026)      # API vs DB 행수
# compare_v2_v3(db_info, "278470", 2026)                   # 에이피알 2026 영업이익·당기순이익 복구 확인
# check_unused_collisions(db_info)                         # 전 종목 충돌 규모


In [4]:
# ==========================================================
# [신규] DB 상태 확인 (테이블 전체 현황 요약)
#  - 실행 전제: Cell 1 (db_info, TABLE_NAME) 실행 완료
# ==========================================================
import pymysql
import pandas as pd


def check_db_status(db_info: dict, table_name: str):
    """테이블 존재 여부, 전체 행 수, 연도x분기 커버리지, 최신 적재 상태를 요약."""
    conn = pymysql.connect(
        host=db_info["host"], port=db_info["port"],
        user=db_info["user"], password=db_info["password"],
        database=db_info["database"], charset="utf8mb4",
        connect_timeout=10,
    )
    try:
        cur = conn.cursor()

        # 0) 테이블 존재 여부
        cur.execute("""
            SELECT COUNT(*) FROM information_schema.tables
            WHERE table_schema = %s AND table_name = %s
        """, (db_info["database"], table_name))
        if cur.fetchone()[0] == 0:
            print(f"❌ 테이블 {table_name} 이 존재하지 않습니다. 수집을 먼저 실행하세요.")
            return

        print("=" * 70)
        print(f"DB 상태 확인: {db_info['database']}.{table_name}")
        print("=" * 70)

        # 1) 전체 행 수 / 티커 수 / 기간
        cur.execute(f"""
            SELECT COUNT(*), COUNT(DISTINCT ticker),
                   MIN(bsns_year), MAX(bsns_year), MAX(report_date)
            FROM {table_name}
        """)
        total_rows, total_tickers, min_y, max_y, max_date = cur.fetchone()
        print(f"\n총 행 수        : {total_rows:,}")
        print(f"총 티커 수      : {total_tickers:,}")
        print(f"수집 연도 범위  : {min_y} ~ {max_y}")
        print(f"최신 report_date: {max_date}")

        # 2) 연도 x 분기 티커 커버리지 (pivot)
        print("\n[연도 x 분기별 티커 수]")
        cur.execute(f"""
            SELECT bsns_year, quarter, COUNT(DISTINCT ticker) AS ticker_cnt
            FROM {table_name}
            GROUP BY bsns_year, quarter
        """)
        cov = pd.DataFrame(cur.fetchall(), columns=["bsns_year", "quarter", "ticker_cnt"])
        pivot = cov.pivot(index="bsns_year", columns="quarter", values="ticker_cnt")
        pivot = pivot.reindex(columns=[c for c in ["Q1", "H1", "Q3", "FY"] if c in pivot.columns])
        print(pivot.fillna(0).astype(int).to_string())

        # 3) 최근 2개 연도 상세 (행 수 포함)
        print(f"\n[최근 연도 상세: {max_y - 1} ~ {max_y}]")
        cur.execute(f"""
            SELECT bsns_year, quarter, reprt_code,
                   COUNT(DISTINCT ticker) AS ticker_cnt, COUNT(*) AS row_cnt,
                   MAX(report_date) AS max_date
            FROM {table_name}
            WHERE bsns_year >= %s
            GROUP BY bsns_year, quarter, reprt_code
            ORDER BY bsns_year, reprt_code
        """, (max_y - 1,))
        detail = pd.DataFrame(cur.fetchall(), columns=[
            "bsns_year", "quarter", "reprt_code", "ticker_cnt", "row_cnt", "max_date"
        ])
        print(detail.to_string(index=False))

        # 4) 누락 경고: 직전 연도 대비 최신 연도 커버리지 급감 여부
        print("\n[커버리지 진단]")
        for q in ["Q1", "H1", "Q3", "FY"]:
            prev = int(pivot.loc[max_y - 1, q]) if (max_y - 1 in pivot.index and q in pivot.columns and pd.notnull(pivot.loc[max_y - 1, q])) else 0
            curr = int(pivot.loc[max_y, q]) if (max_y in pivot.index and q in pivot.columns and pd.notnull(pivot.loc[max_y, q])) else 0
            if prev == 0 and curr == 0:
                continue
            flag = "⚠️ 수집 부족 의심" if curr < prev * 0.9 else "✅"
            print(f"  {q}: {max_y - 1}년 {prev}개 → {max_y}년 {curr}개  {flag}")
        print("\n※ 최신 연도의 미제출 보고서(예: 8월 시점의 Q3/FY)는 적은 것이 정상입니다.")

        print("\n" + "=" * 70)
    finally:
        conn.close()


check_db_status(db_info, TABLE_NAME)


❌ 테이블 korea_fs_data_from_DART_V3 이 존재하지 않습니다. 수집을 먼저 실행하세요.


In [5]:
# ==========================================================
# [신규] DART 최신 데이터 증분 수집
#  - 오늘 날짜 기준으로 "이미 제출됐을 보고서" 목록을 자동 결정
#  - DB에 이미 있는 (ticker, year, quarter)는 건너뛰고 누락분만 수집
#  - 실행 전제: Cell 0 (함수 정의), Cell 1 (API_KEY, db_info, TABLE_NAME)
# ==========================================================
import datetime as _dt
import pymysql
import pandas as pd

REPRT_INFO = {
    "11013": ("Q1", "-03-31"),
    "11012": ("H1", "-06-30"),
    "11014": ("Q3", "-09-30"),
    "11011": ("FY", "-12-31"),
}


def get_available_reports(today: _dt.date = None) -> list:
    """
    오늘 날짜 기준으로 제출 완료됐을 (bsns_year, reprt_code, quarter) 목록 반환.
    법정 제출 기한: Q1=5/15, H1=8/14, Q3=11/14, FY=익년 3/31 (여유 2주 반영)
    """
    today = today or _dt.date.today()
    targets = []
    for year in (today.year - 1, today.year):
        deadlines = {
            "11013": _dt.date(year, 5, 31),        # Q1
            "11012": _dt.date(year, 8, 15),        # H1
            "11014": _dt.date(year, 11, 30),       # Q3
            "11011": _dt.date(year + 1, 4, 15),    # FY (익년)
        }
        for code, dl in deadlines.items():
            if today >= dl:
                targets.append((year, code, REPRT_INFO[code][0]))
    return targets


def get_existing_tickers(db_info: dict, table_name: str,
                         year: int, quarter: str) -> set:
    """해당 (year, quarter)에 이미 데이터가 있는 티커 집합."""
    conn = pymysql.connect(
        host=db_info["host"], port=db_info["port"],
        user=db_info["user"], password=db_info["password"],
        database=db_info["database"], charset="utf8mb4",
    )
    try:
        cur = conn.cursor()
        cur.execute("""
            SELECT COUNT(*) FROM information_schema.tables
            WHERE table_schema = %s AND table_name = %s
        """, (db_info["database"], table_name))
        if cur.fetchone()[0] == 0:
            return set()
        cur.execute(
            f"SELECT DISTINCT ticker FROM {table_name} "
            f"WHERE bsns_year = %s AND quarter = %s",
            (year, quarter),
        )
        return {r[0] for r in cur.fetchall()}
    finally:
        conn.close()


def fetch_one_report(api_key: str, corp_code: str,
                     year: int, reprt_code: str, fs_div: str) -> list:
    """특정 (연도, 보고서, fs_div) 하나만 조회. 행 리스트 반환."""
    quarter_label, date_suffix = REPRT_INFO[reprt_code]
    url = "https://opendart.fss.or.kr/api/fnlttSinglAcntAll.json"
    params = {
        "crtfc_key": api_key, "corp_code": corp_code,
        "bsns_year": str(year), "reprt_code": reprt_code, "fs_div": fs_div,
    }
    time.sleep(0.08)

    data, last_exc = None, None
    for attempt in range(3):
        try:
            r = requests.get(url, params=params, timeout=30)
            r.raise_for_status()
            data = r.json()
            break
        except (requests.exceptions.RequestException, ValueError) as e:
            last_exc = e
            if attempt < 2:
                time.sleep(2 ** attempt)
    if data is None:
        logger.error(f"[SKIP] corp={corp_code} {year}/{reprt_code}: {last_exc}")
        return []

    status = data.get("status")
    if status == "020":
        raise RuntimeError("[RATE LIMIT] DART 일일 호출 한도 초과 (status=020). 수집 중단.")
    if status != "000":
        return []   # 013 = 자료 없음

    rows = []
    for item in data.get("list", []):
        rows.append({
            "corp_code": item.get("corp_code"),
            "bsns_year": int(item.get("bsns_year")),
            "reprt_code": item.get("reprt_code"),
            "sj_div": item.get("sj_div"),
            "sj_nm": item.get("sj_nm"),
            "account_id": item.get("account_id"),
            "account_nm": item.get("account_nm"),
            "thstrm_nm": item.get("thstrm_nm"),
            "thstrm_amount": item.get("thstrm_amount"),
            "thstrm_add_amount": item.get("thstrm_add_amount"),   # ★ v7
            "ord": item.get("ord"),                               # ★ v7
            "fs_div": fs_div,                                     # ★ v7
            "quarter": quarter_label,
            "report_date": _dt.datetime.strptime(f"{year}{date_suffix}", "%Y-%m-%d").date(),
        })
    return rows


def collect_latest_dart_data(api_key: str, db_info: dict,
                             table_name: str = FS_TABLE,
                             batch_size: int = 20,
                             max_companies: int = None):
    """
    최신 보고서 중 DB에 누락된 (티커 x 보고서)만 골라 수집·저장.
    - CFS 시도 후 자료 없으면 OFS fallback (보고서 단위)
    - batch_size개 회사 단위로 DB 저장
    - max_companies: 테스트용 상한 (None = 전체)
    """
    if not test_db_connection(db_info):
        logger.error("DB 연결 실패로 작업 중단")
        return []

    targets = get_available_reports()
    if not targets:
        print("현재 제출 완료된 최신 보고서가 없습니다.")
        return []

    print("=" * 70)
    print("[증분 수집 대상 보고서]")
    for y, code, q in targets:
        print(f"  {y}년 {q} (reprt_code={code})")
    print("=" * 70)

    # 상장사 목록
    corp_df = load_corp_code(api_key)
    corp_df = corp_df[
        corp_df["stock_code"].notna() &
        (corp_df["stock_code"].astype(str).str.strip() != "")
    ].copy()
    corp_df["stock_code"] = corp_df["stock_code"].astype(str).str.zfill(6)
    corp_df = corp_df.sort_values("stock_code").reset_index(drop=True)
    print(f"상장사 수: {len(corp_df)}개")

    # 보고서별 누락 티커 계산
    all_tickers = set(corp_df["stock_code"])
    missing_map = {}   # (year, reprt_code, quarter) -> 누락 티커 set
    for y, code, q in targets:
        existing = get_existing_tickers(db_info, table_name, y, q)
        missing = all_tickers - existing
        missing_map[(y, code, q)] = missing
        print(f"  {y} {q}: 기존 {len(existing)}개 / 누락 {len(missing)}개")

    # 티커 → 수집해야 할 보고서 목록
    ticker_jobs = {}
    for key, missing in missing_map.items():
        for t in missing:
            ticker_jobs.setdefault(t, []).append(key)

    job_tickers = sorted(ticker_jobs.keys())
    if max_companies:
        job_tickers = job_tickers[:max_companies]
    total = len(job_tickers)
    print(f"\n수집 대상 기업 수: {total}개")
    if total == 0:
        print("🎉 모든 최신 데이터가 이미 수집되어 있습니다.")
        return []

    corp_map = dict(zip(corp_df["stock_code"],
                        zip(corp_df["corp_code"], corp_df["corp_name"])))

    target_cols = TARGET_COLS   # ★ v7

    error_list, batch_list, batch_codes = [], [], []
    saved_companies = 0

    for i, ticker in enumerate(job_tickers, 1):
        corp_code, corp_name = corp_map[ticker]
        jobs = ticker_jobs[ticker]

        if i % 50 == 1 or i == total:
            logger.info(f"[{i}/{total}] {corp_name}({ticker}) — 보고서 {len(jobs)}건")

        try:
            rows = []
            for (y, code, q) in jobs:
                r = fetch_one_report(api_key, corp_code, y, code, fs_div="CFS")
                if not r:
                    r = fetch_one_report(api_key, corp_code, y, code, fs_div="OFS")
                rows.extend(r)

            if not rows:
                # 미제출 기업(신규상장 등)일 수 있으므로 에러로만 기록
                error_list.append((ticker, corp_name, "자료 없음 (CFS/OFS 모두)"))
                continue

            df = pd.DataFrame(rows)
            df["thstrm_amount"] = pd.to_numeric(df["thstrm_amount"], errors="coerce")
            df = df[target_cols].copy()
            df["ticker"] = ticker

            batch_list.append(df)
            batch_codes.append(ticker)

            if len(batch_list) >= batch_size:
                save_fs_batch_to_db(batch_list, db_info=db_info, table_name=table_name)
                saved_companies += len(batch_list)
                logger.info(f"[BATCH SAVE] 누적 {saved_companies}개 회사 저장 완료")
                batch_list, batch_codes = [], []

        except RuntimeError as re_err:
            # 일일 한도 초과 → 지금까지 모은 배치 저장 후 중단
            logger.error(str(re_err))
            if batch_list:
                save_fs_batch_to_db(batch_list, db_info=db_info, table_name=table_name)
                saved_companies += len(batch_list)
            print(f"\n⚠️ 일일 한도 초과로 중단. 내일 이 셀을 다시 실행하면 이어서 수집됩니다.")
            return error_list
        except Exception as e:
            logger.error(f"{corp_name}({ticker}) 오류: {e}")
            error_list.append((ticker, corp_name, str(e)))

    if batch_list:
        save_fs_batch_to_db(batch_list, db_info=db_info, table_name=table_name)
        saved_companies += len(batch_list)

    print("\n" + "=" * 70)
    print(f"[증분 수집 완료] 저장 {saved_companies}개 / 자료없음·에러 {len(error_list)}개")
    if error_list:
        print("\n[에러/자료없음 상위 20개]")
        for t, n, m in error_list[:20]:
            print(f"  {t} / {n} / {m[:80]}")
    print("=" * 70)
    return error_list


# ---- 실행 ----
# 먼저 소규모 테스트를 권장합니다: max_companies=20
# 문제 없으면 max_companies=None 으로 전체 실행 (중단돼도 재실행 시 이어서 수집됨)
latest_errors = collect_latest_dart_data(
    api_key=API_KEY,
    db_info=db_info,
    table_name=TABLE_NAME,
    batch_size=20,
    max_companies=None,   # 테스트 후 None으로 변경
)


2026-08-25 18:06:38 [INFO] DB 연결 성공


[증분 수집 대상 보고서]
  2025년 Q1 (reprt_code=11013)
  2025년 H1 (reprt_code=11012)
  2025년 Q3 (reprt_code=11014)
  2025년 FY (reprt_code=11011)
  2026년 Q1 (reprt_code=11013)
  2026년 H1 (reprt_code=11012)


2026-08-25 18:06:41 [INFO] [1/3986] 신한은행(000010) — 보고서 6건


상장사 수: 3986개
  2025 Q1: 기존 0개 / 누락 3986개
  2025 H1: 기존 0개 / 누락 3986개
  2025 Q3: 기존 0개 / 누락 3986개
  2025 FY: 기존 0개 / 누락 3986개
  2026 Q1: 기존 0개 / 누락 3986개
  2026 H1: 기존 0개 / 누락 3986개

수집 대상 기업 수: 3986개


2026-08-25 18:07:16 [WARNING] [BATCH] 배치 내 PK 중복 제거: 11857 rows
2026-08-25 18:07:17 [ERROR] [BATCH ERROR] (1406, "Data too long for column 'account_id' at row 1646")
2026-08-25 18:07:17 [ERROR] 삼천당제약(000250) 오류: (1406, "Data too long for column 'account_id' at row 1646")
2026-08-25 18:07:19 [WARNING] [BATCH] 배치 내 PK 중복 제거: 12356 rows
2026-08-25 18:07:20 [ERROR] [BATCH ERROR] (1406, "Data too long for column 'account_id' at row 1646")
2026-08-25 18:07:20 [ERROR] 기아(000270) 오류: (1406, "Data too long for column 'account_id' at row 1646")
2026-08-25 18:07:21 [WARNING] [BATCH] 배치 내 PK 중복 제거: 12510 rows
2026-08-25 18:07:22 [ERROR] [BATCH ERROR] (1406, "Data too long for column 'account_id' at row 1646")
2026-08-25 18:07:22 [ERROR] DH오토넥스(000300) 오류: (1406, "Data too long for column 'account_id' at row 1646")
2026-08-25 18:07:24 [WARNING] [BATCH] 배치 내 PK 중복 제거: 12937 rows
2026-08-25 18:07:24 [ERROR] [BATCH ERROR] (1406, "Data too long for column 'account_id' at row 1646")
2026-08-25 18:07:24 

KeyboardInterrupt: 

In [6]:
import pymysql

conn = pymysql.connect(
    host=db_info["host"], port=db_info["port"],
    user=db_info["user"], password=db_info["password"],
    database=db_info["database"], charset="utf8mb4",
)
try:
    with conn.cursor() as cur:
        cur.execute(f"DROP TABLE IF EXISTS {TABLE_NAME}")
    conn.commit()
    print(f"[DROP] {TABLE_NAME} 삭제 완료 (없었으면 무시됨)")
finally:
    conn.close()

[DROP] korea_fs_data_from_DART_V3 삭제 완료 (없었으면 무시됨)


In [17]:
# (선택) 수집된 계정과목 목록 확인
test['account_nm'].unique().tolist()


['계약자산',
 '관계기업의 취득',
 '관계기업투자주식',
 '금융비용',
 '금융수익',
 '기말의 현금및현금성자산',
 '기본주당이익',
 '기초의 현금및현금성자산',
 '기초자본',
 '기타금융채무',
 '기타비용',
 '기타비유동부채',
 '기타비유동자산',
 '기타수익',
 '기타수취채권',
 '기타유동부채',
 '기타유동자산',
 '기타자본항목',
 '기타지급채무',
 '기타포괄손익-공정가치 측정 지분상품 처분에 따른 이익잉여금 대체',
 '기타포괄손익-공정가치금융자산평가손익',
 '기타포괄손익-공정가치측정금융자산',
 '기타포괄손익-공정가치측정금융자산감소',
 '기타포괄손익누계액',
 '납입자본',
 '단기금융상품',
 '단기금융상품의 감소',
 '단기대여금의 감소',
 '단기대여금의 증가',
 '당기법인세부채',
 '당기법인세자산',
 '당기손익-공정가치측정금융부채',
 '당기손익-공정가치측정금융자산',
 '당기손익-공정가치측정금융자산 감소',
 '당기손익-공정가치측정금융자산 증가',
 '리스부채',
 '리스부채의 상환',
 '리스사용권자산',
 '매입채무',
 '매출액',
 '매출원가',
 '매출채권',
 '매출총이익',
 '무형자산',
 '무형자산의 처분',
 '무형자산의 취득',
 '배당금수취',
 '배당금지급',
 '법인세비용',
 '법인세비용차감전순이익',
 '법인세의 납부',
 '보증금의 감소',
 '보증금의 증가',
 '부채와 자본총계',
 '부채총계',
 '비유동부채',
 '비유동자산',
 '비지배지분',
 '사업결합',
 '사업결합으로 인한 현금흐름',
 '세후 기타포괄손익',
 '세후 총포괄손익',
 '순확정급여부채',
 '순확정급여부채의 재측정요소',
 '연결당기순이익',
 '영업으로부터 창출된 현금흐름',
 '영업이익',
 '영업활동으로 인한 현금흐름',
 '유동부채',
 '유동자산',
 '유형자산',
 '유형자산의 처분',
 '유형자산의 취득',
 '이연법인세부채',
 '이연법인세자산',
 '이익잉여금',
 '이자의 수

In [18]:
# ==========================================================
# 실행: 회계연도 재무데이터 수집 (전 상장사, 전체 재수집)
#  ★ v7: V3 테이블은 비어 있으므로 START_YEAR 를 2015 등으로 낮춰 전체 재적재 권장
#     (일일 한도 초과 시 status=020 에서 중단 → 다음날 증분 셀(Cell 3)로 이어서)
#  - 2026-08 기준 존재하는 최신 보고서:
#    FY2025(11011), 2026 Q1(11013), 2026 H1(11012)
#  ※ 전체 재수집용. 이미 수집된 티커를 건너뛰는 "증분 수집"은
#    아래 새로 추가된 셀(collect_latest_dart_data)을 사용하세요.
# ==========================================================
# - 자동으로 500개씩 배치 분할 실행 → rate limit 회피
# - 1차 실행 후 실패한 티커는 자동으로 재시도
# - 전체 소요 시간: 약 1~3시간 (기업 수 및 네트워크 상황에 따라)

from datetime import datetime

START_YEAR = 2015   # ★ v7 전체 재적재 (테스트는 2025 로)
END_YEAR   = 2026   # 2026 Q1/H1 포함
BATCH_CHUNK = 500   # 한 번에 처리할 기업 수 (500개 단위)

t_start = datetime.now()
print(f"[시작] {t_start:%Y-%m-%d %H:%M:%S}  — {START_YEAR} 회계연도 수집")
print("=" * 70)

# ---- 1차 실행: 500개씩 분할 ----
all_errors = []
for chunk_start in range(0, 99999, BATCH_CHUNK):
    chunk_end = chunk_start + BATCH_CHUNK
    print(f"\n>>> Chunk [{chunk_start}:{chunk_end}) 시작 ({datetime.now():%H:%M:%S})")

    try:
        chunk_errors = run_dart_fs_for_top_range(
            api_key=API_KEY,
            db_info=db_info,
            start_year=START_YEAR,
            end_year=END_YEAR,
            top_start=chunk_start,
            top_end=chunk_end,
            batch_size=10,
            table_name=TABLE_NAME,
        )
        all_errors.extend(chunk_errors)
    except Exception as e:
        print(f"[CHUNK FATAL] {chunk_start}:{chunk_end} 중단 — {e}")

    # 실제 chunk가 비어있으면 (전체 상장사 범위 벗어남) 종료
    # run_dart_fs_for_top_range가 빈 리스트를 반환하면 다음 chunk는 불필요
    # 간단한 heuristic: chunk_start >= 4000이면 한국 전 상장사를 넘은 것으로 간주
    if chunk_start >= 4000:
        break

# ---- 2차 실행: 실패 티커만 재시도 ----
# "재무데이터 없음"은 진짜로 DART에 없는 경우가 많으므로 제외하고,
# 네트워크/타임아웃/BATCH_ERROR 등 transient 에러만 재시도
transient_errors = [
    e for e in all_errors
    if e[1] not in ("NOT_FOUND_IN_DART",)
    and "재무데이터 없음" not in str(e[2])
]
retry_tickers = sorted(set([e[0] for e in transient_errors]))

retry_errors = []
if retry_tickers:
    print(f"\n{'='*70}")
    print(f"[재시도] transient 에러 {len(retry_tickers)}개 티커 재시도")
    print(f"{'='*70}")

    retry_errors = run_dart_fs_for_stock_list(
        api_key=API_KEY,
        db_info=db_info,
        stock_code_list=retry_tickers,
        start_year=START_YEAR,
        end_year=END_YEAR,
        batch_size=10,
        table_name=TABLE_NAME,
    )

# ---- 최종 결과 요약 ----
t_end = datetime.now()
elapsed = t_end - t_start

print(f"\n{'='*70}")
print(f"[완료] {t_end:%Y-%m-%d %H:%M:%S}  — 소요시간: {elapsed}")
print(f"{'='*70}")
print(f"1차 에러: {len(all_errors)}개")
print(f"재시도 대상: {len(retry_tickers)}개")
print(f"재시도 후 남은 실패: {len(retry_errors)}개")
print()

# 최종 실패 목록 상세 출력 (상위 30개)
if retry_errors:
    print(f"\n[최종 실패 티커 - 상위 30개]")
    for sc, name, msg in retry_errors[:30]:
        print(f"  {sc} / {name} / {msg[:80]}")

    # 실패 목록 DataFrame으로 보관
    import pandas as pd
    final_error_df = pd.DataFrame(retry_errors, columns=['ticker', 'corp_name', 'error_msg'])
    print(f"\nfinal_error_df 변수에 {len(final_error_df)}개 실패 티커 저장됨")
else:
    print("\n🎉 모든 티커가 성공적으로 수집되었습니다!")
    final_error_df = None


[시작] 2026-08-24 10:59:39  — 2025 회계연도 수집

>>> Chunk [0:500) 시작 (10:59:39)


2026-08-24 10:59:44 [INFO] DB 연결 성공
2026-08-24 10:59:47 [INFO] DART corp 리스트 [0:500) 구간 선택: 전체 118747개 -> 대상 500개
2026-08-24 10:59:47 [INFO] DB 연결 성공
2026-08-24 10:59:47 [INFO] DB 연결 테스트 완료
2026-08-24 10:59:47 [INFO] [STEP 1] DART 기업 목록 로드 중...


[INFO] stock_code 정렬 기준 0~499번째 기업 수: 500


2026-08-24 10:59:49 [INFO] DART 상장사 필터링 완료: 3985개
2026-08-24 10:59:49 [INFO] 사용자 지정 종목 수: 500개 -> 정규화 후 1개
2026-08-24 10:59:49 [WARNING] [1/1] 00000 : DART corp_code를 찾을 수 없음
2026-08-24 10:59:49 [INFO] 작업 완료. 지정 종목 수: 1, 에러 종목 수: 1
2026-08-24 10:59:49 [INFO] DB 연결 성공



[에러 발생 종목 목록]
 - 00000  / NOT_FOUND_IN_DART / DART corp_code를 찾을 수 없음

>>> Chunk [500:1000) 시작 (10:59:49)


...


[에러 발생 종목 목록]
 - 00000  / NOT_FOUND_IN_DART / DART corp_code를 찾을 수 없음

>>> Chunk [4000:4500) 시작 (11:00:23)


2026-08-24 11:00:26 [INFO] DART corp 리스트 [4000:4500) 구간 선택: 전체 118747개 -> 대상 500개
2026-08-24 11:00:26 [INFO] DB 연결 성공
2026-08-24 11:00:26 [INFO] DB 연결 테스트 완료
2026-08-24 11:00:26 [INFO] [STEP 1] DART 기업 목록 로드 중...


[INFO] stock_code 정렬 기준 4000~4499번째 기업 수: 500


2026-08-24 11:00:28 [INFO] DART 상장사 필터링 완료: 3985개
2026-08-24 11:00:28 [INFO] 사용자 지정 종목 수: 500개 -> 정규화 후 1개
2026-08-24 11:00:28 [WARNING] [1/1] 00000 : DART corp_code를 찾을 수 없음
2026-08-24 11:00:28 [INFO] 작업 완료. 지정 종목 수: 1, 에러 종목 수: 1



[에러 발생 종목 목록]
 - 00000  / NOT_FOUND_IN_DART / DART corp_code를 찾을 수 없음

[완료] 2026-08-24 11:00:28  — 소요시간: 0:00:48.960888
1차 에러: 9개
재시도 대상: 0개
재시도 후 남은 실패: 0개


🎉 모든 티커가 성공적으로 수집되었습니다!


In [19]:
# ==========================================================
# 검증: 2025 회계연도 데이터 적재 확인
# ==========================================================
# 수집 작업이 끝난 후 이 셀을 실행하면 DB 적재 현황을 요약해서 보여줍니다.

import pymysql
import pandas as pd

def verify_dart_loading(db_info: dict, bsns_year: int, table_name: str):
    """수집된 재무데이터의 적재 상태를 검증한다."""

    conn = pymysql.connect(
        host=db_info["host"], port=db_info["port"],
        user=db_info["user"], password=db_info["password"],
        database=db_info["database"], charset="utf8mb4",
    )

    try:
        cur = conn.cursor()

        print("=" * 70)
        print(f"DB 적재 검증 - {bsns_year} 회계연도 / 테이블: {table_name}")
        print("=" * 70)

        # ---- 1. 보고서별 티커 수와 행 수 ----
        print("\n[1] 분기별 적재 현황")
        cur.execute(f"""
            SELECT
                quarter,
                reprt_code,
                COUNT(DISTINCT ticker) AS ticker_cnt,
                COUNT(*)               AS row_cnt,
                MIN(report_date)       AS min_date,
                MAX(report_date)       AS max_date
            FROM {table_name}
            WHERE bsns_year = %s
            GROUP BY quarter, reprt_code
            ORDER BY reprt_code
        """, (bsns_year,))
        rows = cur.fetchall()

        if not rows:
            print(f"  ⚠️  {bsns_year}년 데이터가 아예 없습니다. 수집이 실패했을 가능성.")
            return

        summary_df = pd.DataFrame(rows, columns=[
            'quarter', 'reprt_code', 'ticker_cnt', 'row_cnt', 'min_date', 'max_date'
        ])
        print(summary_df.to_string(index=False))

        total_tickers = summary_df['ticker_cnt'].max()
        print(f"\n  → 4개 보고서 중 가장 많은 티커가 있는 보고서: {total_tickers}개")

        # ---- 2. 매출 데이터 존재 여부 (예측 파이프라인 연결용) ----
        print("\n[2] 매출 데이터 존재 티커 수 (account_id='ifrs-full_Revenue' or 'ifrs_Revenue')")
        cur.execute(f"""
            SELECT
                quarter,
                COUNT(DISTINCT ticker) AS rev_ticker_cnt
            FROM {table_name}
            WHERE bsns_year = %s
              AND account_id IN ('ifrs_Revenue', 'ifrs-full_Revenue')
            GROUP BY quarter
            ORDER BY quarter
        """, (bsns_year,))
        rev_rows = cur.fetchall()
        rev_df = pd.DataFrame(rev_rows, columns=['quarter', 'rev_ticker_cnt'])
        print(rev_df.to_string(index=False))

        # ---- 3. 샘플 티커 확인 (삼성전자, SK하이닉스, NAVER) ----
        print("\n[3] 주요 티커 샘플 확인 - 매출 데이터")
        sample_tickers = ['005930', '000660', '035420']  # 삼전, 하이닉스, NAVER
        placeholders = ','.join(['%s'] * len(sample_tickers))
        cur.execute(f"""
            SELECT ticker, quarter, account_nm, thstrm_amount, report_date
            FROM {table_name}
            WHERE bsns_year = %s
              AND ticker IN ({placeholders})
              AND account_id IN ('ifrs_Revenue', 'ifrs-full_Revenue')
            ORDER BY ticker, report_date
        """, (bsns_year, *sample_tickers))
        sample_rows = cur.fetchall()

        if sample_rows:
            sample_df = pd.DataFrame(sample_rows, columns=[
                'ticker', 'quarter', 'account_nm', 'thstrm_amount', 'report_date'
            ])
            # 금액을 억원 단위로 환산
            sample_df['억원'] = (sample_df['thstrm_amount'].astype(float) / 1e8).round(0).astype(int)
            print(sample_df[['ticker', 'quarter', 'account_nm', '억원', 'report_date']].to_string(index=False))
        else:
            print("  ⚠️  샘플 티커의 매출 데이터가 없습니다.")

        # ---- 4. 이전 연도 대비 누락 티커 확인 ----
        print("\n[4] 이전 연도 대비 누락 티커 확인")
        cur.execute(f"""
            SELECT COUNT(DISTINCT ticker)
            FROM {table_name}
            WHERE bsns_year = %s AND quarter = 'Q1'
        """, (bsns_year - 1,))
        prev_cnt = cur.fetchone()[0]

        cur.execute(f"""
            SELECT COUNT(DISTINCT ticker)
            FROM {table_name}
            WHERE bsns_year = %s AND quarter = 'Q1'
        """, (bsns_year,))
        curr_cnt = cur.fetchone()[0]

        print(f"  {bsns_year - 1}년 Q1 티커 수: {prev_cnt}")
        print(f"  {bsns_year}년 Q1 티커 수: {curr_cnt}")
        print(f"  차이: {curr_cnt - prev_cnt:+d} "
              f"({'수집 부족 의심' if curr_cnt < prev_cnt * 0.9 else '정상 범위'})")

        # ---- 5. 중복 검사 ----
        print("\n[5] 중복 레코드 검사 (동일 ticker+date+account_id 조합)")
        cur.execute(f"""
            SELECT COUNT(*) FROM (
                SELECT ticker, report_date, account_id, COUNT(*) AS c
                FROM {table_name}
                WHERE bsns_year = %s
                GROUP BY ticker, report_date, account_id
                HAVING c > 1
            ) dup
        """, (bsns_year,))
        dup_cnt = cur.fetchone()[0]
        if dup_cnt > 0:
            print(f"  ⚠️  중복 의심 조합: {dup_cnt}개 "
                  f"(UNIQUE KEY 미설정 시 발생 가능. 원인 점검 필요)")
        else:
            print(f"  ✅ 중복 없음")

        print("\n" + "=" * 70)
        print("검증 완료")
        print("=" * 70)

    finally:
        conn.close()


# 검증 실행 (수집 범위의 모든 연도 확인)
for _y in range(START_YEAR, END_YEAR + 1):
    verify_dart_loading(db_info, bsns_year=_y, table_name=TABLE_NAME)


DB 적재 검증 - 2025 회계연도 / 테이블: korea_fs_data_from_DART_V2

[1] 분기별 적재 현황
quarter reprt_code  ticker_cnt  row_cnt   min_date   max_date
     FY      11011        2788   315490 2025-12-31 2025-12-31
     H1      11012        2745   281204 2025-06-30 2025-06-30
     Q1      11013        2708   265854 2025-03-31 2025-03-31
     Q3      11014        2727   286352 2025-09-30 2025-09-30

  → 4개 보고서 중 가장 많은 티커가 있는 보고서: 2788개

[2] 매출 데이터 존재 티커 수 (account_id='ifrs-full_Revenue' or 'ifrs_Revenue')
quarter  rev_ticker_cnt
     FY            2665
     H1            2626
     Q1            2587
     Q3            2607

[3] 주요 티커 샘플 확인 - 매출 데이터
ticker quarter account_nm      억원 report_date
000660      Q1        매출액  176391  2025-03-31
000660      H1        매출액  222320  2025-06-30
000660      Q3        매출액  244489  2025-09-30
000660      FY        매출액  971467  2025-12-31
005930      Q1        매출액  791405  2025-03-31
005930      H1        매출액  745663  2025-06-30
005930      Q3        매출액  860617  2025-09-

In [20]:
# ==========================================================
# FY 누락 원인 진단 쿼리
# ==========================================================
import pymysql
import pandas as pd

conn = pymysql.connect(
    host=db_info["host"], port=db_info["port"],
    user=db_info["user"], password=db_info["password"],
    database=db_info["database"], charset="utf8mb4",
)

try:
    # ─────────────────────────────────────────────────
    # ① FY가 있는 37개 티커 확인
    # ─────────────────────────────────────────────────
    print("=" * 70)
    print("[①] 2025 FY가 수집된 티커 목록 (최대 50개)")
    print("=" * 70)
    sql1 = """
        SELECT DISTINCT ticker, corp_code
        FROM korea_fs_data_from_DART_V2
        WHERE bsns_year = 2025 AND quarter = 'FY'
        ORDER BY ticker
        LIMIT 50
    """
    df1 = pd.read_sql(sql1, conn)
    print(f"반환 행 수: {len(df1)}")
    print(df1.to_string(index=False))

    # ─────────────────────────────────────────────────
    # ② 2024년 FY 수집 상태 비교
    # ─────────────────────────────────────────────────
    print("\n" + "=" * 70)
    print("[②] 2024년 분기별 티커 수 (정상이었다면 FY도 2000+개여야 함)")
    print("=" * 70)
    sql2 = """
        SELECT quarter, reprt_code, COUNT(DISTINCT ticker) AS ticker_cnt
        FROM korea_fs_data_from_DART_V2
        WHERE bsns_year = 2024
        GROUP BY quarter, reprt_code
        ORDER BY reprt_code
    """
    df2 = pd.read_sql(sql2, conn)
    print(df2.to_string(index=False))

    # ─────────────────────────────────────────────────
    # ③ Q3는 있는데 FY가 없는 기업 수
    # ─────────────────────────────────────────────────
    print("\n" + "=" * 70)
    print("[③] 2025 Q3는 있는데 FY가 없는 티커 수 (FY 누락 규모)")
    print("=" * 70)
    sql3 = """
        SELECT COUNT(DISTINCT t3.ticker) AS missing_fy_cnt
        FROM korea_fs_data_from_DART_V2 t3
        LEFT JOIN (
            SELECT DISTINCT ticker FROM korea_fs_data_from_DART_V2
            WHERE bsns_year = 2025 AND quarter = 'FY'
        ) fy ON t3.ticker = fy.ticker
        WHERE t3.bsns_year = 2025
          AND t3.quarter = 'Q3'
          AND fy.ticker IS NULL
    """
    df3 = pd.read_sql(sql3, conn)
    print(df3.to_string(index=False))

    # ─────────────────────────────────────────────────
    # ④ 2024년 FY는 있는데 2025년 FY가 없는 티커
    # ─────────────────────────────────────────────────
    print("\n" + "=" * 70)
    print("[④] 2024년 FY는 있는데 2025년 FY가 없는 티커 수 (진짜 누락)")
    print("=" * 70)
    sql4 = """
        SELECT COUNT(DISTINCT t24.ticker) AS real_missing_cnt
        FROM korea_fs_data_from_DART_V2 t24
        LEFT JOIN (
            SELECT DISTINCT ticker FROM korea_fs_data_from_DART_V2
            WHERE bsns_year = 2025 AND quarter = 'FY'
        ) t25 ON t24.ticker = t25.ticker
        WHERE t24.bsns_year = 2024
          AND t24.quarter = 'FY'
          AND t25.ticker IS NULL
    """
    df4 = pd.read_sql(sql4, conn)
    print(df4.to_string(index=False))

finally:
    conn.close()

print("\n" + "=" * 70)
print("진단 완료")
print("=" * 70)

[①] 2025 FY가 수집된 티커 목록 (최대 50개)
반환 행 수: 50
ticker corp_code
000010  00149293
000020  00119195
000030  00254045
000040  00112378
000050  00101628
000060  00117744
000070  00126937
000080  00150244
000100  00145109
000110  00148504
000120  00113410
000140  00148993
000150  00117212
000180  00133335
0001A0  01516933
000210  00109693
000220  00144818
000230  00146083
000240  00160047
000250  00128546
000270  00106641
000300  00126089
000320  00113508
000370  00135917
000390  00129387
000400  00113562
000430  00111847
000440  00149770
000480  00148984
000490  00109286
0004V0  00924931
0004Y0  01884065
000500  00104768
000520  00128032
000540  00103176
000590  00149026
000640  00116824
000650  00151395
000660  00164779
000670  00141307
000680  00104698
000700  00163691
000720  00164478
000760  00145686
0007C0  01829819
0007J0  01869710
000800  00101257
000810  00139214
000850  00166519
000860  00100939

[②] 2024년 분기별 티커 수 (정상이었다면 FY도 2000+개여야 함)
quarter reprt_code  ticker_cnt
     FY      11